In [33]:
import pandas as pd
import soundfile as sf
import librosa
import os
import numpy as np

In [34]:
folder=r"C:\Users\Lenovo\Desktop\Mindcloud\Final project\ML model\initial"
list=[]
for root,dirs,files in os.walk(folder):
    for file in files:
        filepath=os.path.join(root,file)
        folder_name = os.path.basename(root)
        list.append({"Category":folder_name,"File name":file,"Path":filepath})
initialData=pd.DataFrame(list)

In [35]:
from sklearn.model_selection import train_test_split
trainD,testD=train_test_split(initialData, test_size= 0.2, stratify= initialData["Category"],random_state=42)
testD.to_csv("Test.csv")

In [36]:
count=trainD["Category"].value_counts()
target=count.max()
atrainD=[]
pitch=[-2,1,-1,2]
speed=[0.9,1.1]
noise=[0.01,0.02,0.03]
newdir="./Augcopy2"
os.makedirs(newdir,exist_ok=1)

In [37]:
def addnoise(aud,noisefactor):
    noise=np.random.normal(0,aud.std(),aud.size)
    noiseaud=aud+noise*noisefactor
    return noiseaud
def pitchshift(aud,p,sr):
    naudio=librosa.effects.pitch_shift(aud,sr=sr,n_steps=p)
    return naudio
def timeshift(aud,s):
    naudio=librosa.effects.time_stretch(aud,rate=s)
    return naudio

In [38]:
for l in trainD["Category"].unique():
    f=trainD[trainD["Category"]==l]
    n=target-len(f)
    if n<=0:
        continue
    for i, row in f.iterrows():
        if n<=0:
            break
        audio,sr=librosa.load(row["Path"],sr=22050)
        for p in pitch:
            if n<=0:
                break
            for s in speed:
                if n<=0:
                    break
                for nl in noise:
                    if n<=0:
                        break
                    naudio=pitchshift(audio,p,sr)
                    naudio=timeshift(naudio,s)
                    naudio=addnoise(naudio,nl)
                    fname=f"{l}{row['File name'][:-4]}_pitch{p}_speed{s}_noise{nl}.wav"
                    fp=f"./Augcopy2/{fname}"
                    sf.write(fp,naudio,sr)
                    atrainD.append({"Category":l,"File name":fname,"Path":fp})
                    n-=1

In [39]:
atrainD2=pd.DataFrame(atrainD)
augtrainig=pd.concat([trainD,atrainD2])
augtrainig.to_csv("Fullaudio3.csv")
augtrainig.groupby("Category").count()

,File name,Path
Category,,
discomfort,305,305
hungry,305,305
tired,305,305
